# Run All BESS Valuation Phases

Execute phase notebooks `01` through `07` sequentially. Executed copies are written to `data/processed/notebook_runs/<timestamp>/` so the source notebooks are not overwritten.

In [ ]:
import sys
from pathlib import Path

# Bootstrap imports from the project root, then use the shared helper.
_ROOT_CANDIDATE = Path.cwd()
for candidate in [_ROOT_CANDIDATE, *_ROOT_CANDIDATE.parents]:
    if (candidate / "src").is_dir() and (candidate / "data").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not find project root containing src/ and data/.")

from src.utils import find_project_root

PROJECT_ROOT = find_project_root(_ROOT_CANDIDATE)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
missing = [name for name in PHASE_NOTEBOOKS if not (NOTEBOOK_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing phase notebooks: {missing}")

for i, name in enumerate(PHASE_NOTEBOOKS, start=1):
    print(f"{i}. {name}")

In [ ]:
results = []
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")

for phase_name in PHASE_NOTEBOOKS:
    src = NOTEBOOK_DIR / phase_name
    out_name = phase_name.replace(".ipynb", ".executed.ipynb")
    cmd = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        "--ExecutePreprocessor.timeout=-1",
        "--ExecutePreprocessor.kernel_name=python3",
        f"--output-dir={OUTPUT_DIR}",
        f"--output={out_name}",
        str(src),
    ]

    print("\n" + "=" * 80)
    print(f"Executing {phase_name}")
    print("=" * 80)
    started = time.time()
    proc = subprocess.run(
        cmd,
        cwd=PROJECT_ROOT,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    elapsed = time.time() - started
    results.append({
        "notebook": phase_name,
        "returncode": proc.returncode,
        "elapsed_s": elapsed,
        "output": out_name,
    })
    print(proc.stdout[-4000:])
    print(f"Finished {phase_name} in {elapsed:.1f}s with return code {proc.returncode}")
    if proc.returncode != 0:
        raise RuntimeError(f"Execution failed: {phase_name}")

print("\nAll phase notebooks completed.")

In [ ]:
import pandas as pd

summary = pd.DataFrame(results)
summary["elapsed_min"] = summary["elapsed_s"] / 60.0
summary_path = OUTPUT_DIR / "run_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved run summary: {summary_path}")
summary